# Notebook 05: RAG (Retrieval-Augmented Generation) Layer

This notebook demonstrates a lightweight RAG pipeline:

1. **Retrieve** — find relevant papers using embeddings (using our recommender)
2. **Augment** — format retrieved papers as LLM context
3. **Generate** — produce natural language summaries and insights

---

### LLM Backend Options

This notebook supports multiple backends:
- **OpenAI** — requires `OPENAI_API_KEY` environment variable
- **Google Gemini** — requires `GOOGLE_API_KEY` environment variable

If no LLM is available, the retrieval step still works — you just won't get generated summaries.

We will be using a sampled subset of 100k as anything more than that would need to compute for 30 minutes and above.

All the LLM responses will be pasted to a markdown cell at the end of this notebook as a part of the summary.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser import load_from_parquet
from src.embeddings import load_or_compute_embeddings
from src.recommender import ContentRecommender

# Load data
df = load_from_parquet(PROJECT_ROOT / "data" / "processed" / "dblp_subset.parquet")
df_rec = df.dropna(subset=['title']).copy()
df_rec = df_rec[df_rec['title'].str.len() > 10].reset_index(drop=True)

RECOMMENDER_SIZE = min(100_000, len(df_rec))
df_rec = df_rec.sample(n=RECOMMENDER_SIZE, random_state=42).reset_index(drop=True)
titles = df_rec['title'].tolist()

# Load embeddings
CACHE_DIR = PROJECT_ROOT / "data" / "processed"
embeddings = load_or_compute_embeddings(
    titles,
    cache_path=CACHE_DIR / "embeddings_recommender.npy",
    method="transformer",
    model_name="all-MiniLM-L6-v2",
)

# Initialize recommender
recommender = ContentRecommender(df=df_rec, embeddings=embeddings)
print(f"RAG system ready with {len(df_rec):,} papers")

Loading cached embeddings from d:\Rekrutacja_nokia\data\processed\embeddings_recommender.npy
Loaded embeddings: shape (100000, 384)
Recommender initialized with 100,000 papers, embedding dim=384
RAG system ready with 100,000 papers


## 6.1 Retrieval Step

Let's see what papers are retrieved for sample queries.

In [2]:
from src.rag import retrieve_context, build_prompt
from dotenv import load_dotenv
load_dotenv()

query = "What are recent trends in graph neural networks?"

results, context = retrieve_context(query, recommender, top_k=5)

print(f"Query: \"{query}\"\n")
print("Retrieved Papers:")
print(context)

print("\n" + "="*70)
print("\nFormatted as DataFrame:")
results[['title', 'year', 'venue', 'similarity_score']]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: "What are recent trends in graph neural networks?"

Retrieved Papers:
[1] Title: On the Bottleneck of Graph Neural Networks and its Practical Implications.
    Authors: Uri Alon 0002, Eran Yahav
    Year: 2020 | Venue: CoRR
    Relevance Score: 0.6679

[2] Title: Graph Neural Networks in Computer Vision - Architectures, Datasets and Common Approaches.
    Authors: Maciej Krzywda, Szymon Lukasik, Amir H. Gandomi
    Year: 2022 | Venue: IJCNN
    Relevance Score: 0.6290

[3] Title: Graph Neural Networks in Computer Vision - Architectures, Datasets and Common Approaches.
    Authors: Maciej Krzywda, Szymon Lukasik, Amir H. Gandomi
    Year: 2022 | Venue: CoRR
    Relevance Score: 0.6290

[4] Title: DropGNN: Random Dropouts Increase the Expressiveness of Graph Neural Networks.
    Authors: Pál András Papp, Karolis Martinkus, Lukas Faber, Roger Wattenhofer
    Year: 2021 | Venue: NeurIPS
    Relevance Score: 0.6106

[5] Title: Scalable Spatiotemporal Graph Neural Networks.
    Author

,title,year,venue,similarity_score
0,On the Bottleneck of Graph Neural Networks and...,2020,CoRR,0.667896
1,Graph Neural Networks in Computer Vision - Arc...,2022,IJCNN,0.629018
2,Graph Neural Networks in Computer Vision - Arc...,2022,CoRR,0.629018
3,DropGNN: Random Dropouts Increase the Expressi...,2021,NeurIPS,0.610567
4,Scalable Spatiotemporal Graph Neural Networks.,2022,CoRR,0.608955


## 6.2 Prompt Construction

The prompt is structured to give the LLM clear context and instructions.

In [3]:
# Build prompts for different tasks
prompt_summarize = build_prompt(query, context, task='summarize')
prompt_trends = build_prompt(query, context, task='trends')

print("=== SUMMARIZE PROMPT ===")
print(prompt_summarize[:500])
print("...\n")

print("=== TRENDS PROMPT ===")
print(prompt_trends[:500])
print("...")

=== SUMMARIZE PROMPT ===
You are a research assistant analyzing computer science publications from DBLP.

Based on the following retrieved papers relevant to the query "What are recent trends in graph neural networks?":

[1] Title: On the Bottleneck of Graph Neural Networks and its Practical Implications.
    Authors: Uri Alon 0002, Eran Yahav
    Year: 2020 | Venue: CoRR
    Relevance Score: 0.6679

[2] Title: Graph Neural Networks in Computer Vision - Architectures, Datasets and Common Approaches.
    Authors: Maciej 
...

=== TRENDS PROMPT ===
You are a research assistant analyzing computer science publications from DBLP.

Based on the following retrieved papers relevant to the query "What are recent trends in graph neural networks?":

[1] Title: On the Bottleneck of Graph Neural Networks and its Practical Implications.
    Authors: Uri Alon 0002, Eran Yahav
    Year: 2020 | Venue: CoRR
    Relevance Score: 0.6679

[2] Title: Graph Neural Networks in Computer Vision - Architectures,

## 6.3 Generate Response with LLM

Choose your LLM backend below. If none is available, the retrieval context
above already provides valuable results.

In [5]:
from src.rag import rag_query

# Choose one of:  'openai', 'gemini'
LLM_BACKEND = 'gemini'  # Change this based on your setup

# Backend-specific kwargs
llm_kwargs = {}
if LLM_BACKEND == 'openai':
    llm_kwargs = {'model': 'gpt-3.5-turbo', 'api_key': os.environ.get('OPENAI_API_KEY')}
elif LLM_BACKEND == 'gemini':
    llm_kwargs = {'model': 'gemini-2.5-flash', 'api_key': os.environ.get('GOOGLE_API_KEY')}

print(f"Using LLM backend: {LLM_BACKEND}")

Using LLM backend: gemini


In [6]:
# Run full RAG pipeline
queries = [
    "What are the dominant research trends in machine learning and artificial intelligence based on recent publications",
    "What are the key research directions and challenges in computer vision based on recent publications?",
    "What are the main research areas in computer science today, and how do emerging topics differ from more established ones?",
]


print("\n" + "="*80)
print(f"QUERY: {queries[0]}")
print("="*80)

result = rag_query(
    queries[0], 
    recommender,
    llm_backend=LLM_BACKEND,
    task='summarize',
    top_k=15,
    **llm_kwargs
)

print(f"\nRetrieved {len(result['retrieved_papers'])} papers")
print(f"\nLLM Response:")
print(result['response'])


QUERY: What are the dominant research trends in machine learning and artificial intelligence based on recent publications


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Retrieved 15 papers

LLM Response:
Based on the provided papers, the dominant research trends in machine learning (ML) and artificial intelligence (AI) can be categorized into several key areas, reflecting a shift from purely theoretical advancements to practical deployment, ethical considerations, and a meta-analysis of the field itself.

Here are the key findings and contributions, highlighting common themes and methodologies:

**1. Focus on Applied AI, Deployment, and MLOps (Operationalization of ML/AI)**
A significant trend is the move towards operationalizing AI and ML, addressing the practical challenges of integrating these technologies into real-world applications.
*   **Current Applied Trends:** An editorial from 2023 [1] specifically highlights "current trends in applied artificial intelligence," indicating a strong emphasis on practical implementations.
*   **Deployment and Operational Challenges:** Papers identify critical hurdles in deploying and operating ML systems in p

In [12]:
print("\n" + "="*80)
print(f"QUERY: {queries[1]}")
print("="*80)

result = rag_query(
    queries[1], 
    recommender,
    llm_backend=LLM_BACKEND,
    task='summarize',
    top_k=15,
    **llm_kwargs
)

print(f"\nRetrieved {len(result['retrieved_papers'])} papers")
print(f"\nLLM Response:")
print(result['response'])


QUERY: What are the key research directions and challenges in computer vision based on recent publications?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Retrieved 15 papers

LLM Response:
Based on the retrieved papers, which include a mix of foundational works and a few more recent publications, the key research directions and challenges in computer vision, particularly from the more recent contributions, can be summarized as follows:

### Key Research Directions

1.  **Integration of Ontology and Machine Learning for Deeper Understanding:** A prominent recent direction involves leveraging structured knowledge to enhance machine learning in computer vision. Paper [3] (2023), "Role of Ontology-Informed Machine Learning in Computer Vision," specifically highlights this trend, indicating a move towards more semantically aware and knowledge-driven visual analysis beyond purely data-driven methods.
2.  **Visual Reasoning and Discovery:** Research is moving beyond mere object recognition to enabling systems to perform complex reasoning from visual data. Paper [10] (2015), "Visual reasoning for discoveries," points to this direction, emphasi

In [13]:
print("\n" + "="*80)
print(f"QUERY: {queries[2]}")
print("="*80)

result = rag_query(
    queries[2], 
    recommender,
    llm_backend=LLM_BACKEND,
    task='summarize',
    top_k=15,
    **llm_kwargs
)

print(f"\nRetrieved {len(result['retrieved_papers'])} papers")
print(f"\nLLM Response:")
print(result['response'])


QUERY: What are the main research areas in computer science today, and how do emerging topics differ from more established ones?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Retrieved 15 papers

LLM Response:
The provided papers offer a diverse perspective on various aspects of computer science, though a significant portion focuses on education, history, and the broader impact of the field rather than directly enumerating current research areas or explicitly contrasting emerging with established topics.

Here's a summary of the key findings, contributions, and common themes:

**Key Findings and Contributions:**

1.  **Trends in Computer Science Research:**
    *   One recent paper [1] directly addresses "Trends in computer science research within European countries," indicating an ongoing effort to track the evolution of research domains. While the specific trends are not detailed in the title, its 2022 publication suggests an analysis of contemporary developments.
    *   Another paper [12] highlights "Research Software Science" and the expansion of "Research Software Engineering" as an area aiming to amplify the impact of research software, suggesting a


## QUERY: What are the dominant research trends in machine learning and artificial intelligence based on recent publications.

LLM Response:
Based on the provided papers, the dominant research trends in machine learning (ML) and artificial intelligence (AI) can be categorized into several key areas, reflecting a shift from purely theoretical advancements to practical deployment, ethical considerations, and a meta-analysis of the field itself.

Here are the key findings and contributions, highlighting common themes and methodologies:

**1. Focus on Applied AI, Deployment, and MLOps (Operationalization of ML/AI)**
A significant trend is the move towards operationalizing AI and ML, addressing the practical challenges of integrating these technologies into real-world applications.
*   **Current Applied Trends:** An editorial from 2023 [1] specifically highlights "current trends in applied artificial intelligence," indicating a strong emphasis on practical implementations.
*   **Deployment and Operational Challenges:** Papers identify critical hurdles in deploying and operating ML systems in practice [5] and understanding implementation challenges, particularly in crucial areas like documentation [12]. This shows a maturity in the field where the focus is not just on model creation but on making models robust and maintainable.
*   **Continuous Development Pipelines:** The concept of a "Pipeline for the Continuous Development of Artificial Intelligence Models" is a current state of research and practice [7], underscoring the importance of MLOps principles for managing the lifecycle of AI models effectively.
*   **Infrastructure and Performance:** Understanding and leveraging the I/O patterns of emerging machine learning analytics [10] points to research focused on optimizing the underlying infrastructure for ML applications, ensuring efficiency and scalability.

**2. Ethics, Governance, and Societal Impact**
There's a growing recognition of the broader societal implications of AI and ML, leading to increased research into ethical frameworks, policy, and governance.
*   **Value-Laden Shifts:** Research highlights "value-laden disciplinary shifts in machine learning" [4], indicating a move beyond purely technical metrics to consider the ethical and social values embedded in or affected by ML systems.
*   **Promises and Perils:** A dedicated minitrack introduces the "Promises and Perils of Artificial Intelligence and Machine Learning," covering critical aspects like disruption, adoption, dehumanization, governance, risk, and compliance [9]. This reflects a comprehensive approach to understanding both the benefits and potential harms.
*   **Policy Development:** The focus extends to regional policy considerations, such as "Artificial intelligence policies in Africa over the next five years" [13], demonstrating a global concern for AI regulation and strategic planning.

**3. Meta-Research, Evolution, and Future Directions**
The field is also engaged in self-reflection, analyzing its own evolution, identifying future research questions, and compiling ongoing research efforts.
*   **Guiding Future Research:** Questions are being posed "to Guide the Future of Artificial Intelligence Research" [3], suggesting a strategic effort to define the next generation of breakthroughs and tackle fundamental challenges.
*   **Evolutionary Stages and Multidisciplinary Nature:** Analysis of the "Evolutionary stages and multidisciplinary nature of artificial intelligence research" [8] provides insight into how the field has developed and its extensive connections to other disciplines. This bibliometric approach helps chart the course of AI.
*   **Conference Collections:** Major conferences like the Benelux Conference on Artificial Intelligence and the Belgian Dutch Conference on Machine Learning [2], as well as upcoming IEEE conferences [11], serve as crucial platforms for presenting diverse, ongoing research, collectively indicating the breadth and activity within the field.
*   **Broader Research Trends:** More generally, trends in computer science research within European countries [14] indirectly capture the overarching academic landscape that includes AI/ML as a dominant subfield.

**4. Efficiency and Optimization**
While less explicitly dominant in these recent papers compared to deployment and ethics, the pursuit of more efficient and effective machine learning remains an underlying theme.
*   **More with Less:** An earlier paper from 2016 discusses "Modern Machine Learning: More with Less, Cheaper and Better" [6], a continuous aspiration in the field to achieve better performance with fewer resources. This general principle underpins much of the technical work.
*   **I/O Pattern Optimization:** The work on I/O patterns [10] also contributes to the goal of making ML analytics more efficient.

**Common Methodologies:**
The papers predominantly employ **qualitative analysis, literature reviews, and conceptual discussions** to identify trends, challenges, and future directions [1, 3, 4, 5, 7, 9, 12, 13]. **Bibliometric analysis** is used to study the evolution and interdisciplinary nature of research [8, 14]. **Conference proceedings** [2, 11] serve as compilations of diverse research efforts, reflecting the state of the art at specific points in time. Technical analysis focused on system performance (e.g., I/O patterns) is also present [10].

In summary, recent AI and ML research trends extend beyond core algorithmic advancements to critically examine the full lifecycle of AI systems from **development and deployment (MLOps)**, through their **ethical implications and governance**, and into **strategic planning for future research**. This comprehensive approach reflects a maturing discipline grappling with its real-world impact and sustainability.



## QUERY: What are the key research directions and challenges in computer vision based on recent publications?

Based on the retrieved papers, which include a mix of foundational works and a few more recent publications, the key research directions and challenges in computer vision, particularly from the more recent contributions, can be summarized as follows:

### Key Research Directions

1.  **Integration of Ontology and Machine Learning for Deeper Understanding:** A prominent recent direction involves leveraging structured knowledge to enhance machine learning in computer vision. Paper [3] (2023), "Role of Ontology-Informed Machine Learning in Computer Vision," specifically highlights this trend, indicating a move towards more semantically aware and knowledge-driven visual analysis beyond purely data-driven methods.
2.  **Visual Reasoning and Discovery:** Research is moving beyond mere object recognition to enabling systems to perform complex reasoning from visual data. Paper [10] (2015), "Visual reasoning for discoveries," points to this direction, emphasizing the goal of allowing computer vision systems to infer relationships, understand contexts, and make novel "discoveries."
3.  **Active Vision and Dynamic Environment Interaction:** There's a clear trend towards moving "Beyond the Static Camera" [11] (2011). This involves developing "Active Vision" systems that can intelligently control sensing, adapt to changing environments, and process dynamic visual information in real-time, rather than passively analyzing static images.
4.  **Continuous Advancement and Application-Specific Research:** The field consistently sees new developments, evidenced by recent conference proceedings like "Computer Vision - ACCV 2020" [8] (2021) and the upcoming "ECCV 2024" [2] (2025). Furthermore, computer vision continues to find application in specific domains, such as robotics ([5], [9], [12]) and human-computer interaction [13].

### Key Challenges

1.  **Bridging the Gap Between Perception and Structured Knowledge:** The work on "Ontology-Informed Machine Learning" [3] implicitly highlights the challenge of effectively integrating abstract, symbolic knowledge (like ontologies) with the raw perceptual data processed by machine learning models to achieve more robust and interpretable visual understanding.
2.  **Developing Robust Visual Reasoning Capabilities:** As indicated by [10], enabling machines to perform genuine "visual reasoning" presents a significant challenge. This involves overcoming limitations in current models to understand complex scenes, infer intentions, and predict future states, moving beyond simple pattern recognition.
3.  **Handling Dynamic and Unconstrained Environments (Active Vision):** The call to move "Beyond the Static Camera" and address "Issues and Trends in Active Vision" [11] identifies challenges related to real-time processing of dynamic scenes, autonomous camera control, adapting to unpredictable changes, and interacting intelligently with the visual world.
4.  **Addressing "Grand Challenges" in Image Understanding:** Paper [14] (2004), "Grand challenges in image processing and analysis," points to fundamental and persistent difficulties in achieving comprehensive and human-like understanding of images. While specific challenges evolve, the underlying pursuit of robust, generalizable, and context-aware image understanding remains a core hurdle.

### Common Themes and Methodologies

*   **Evolution of Knowledge Representation:** From early work on "Representing Knowledge of the Visual World" [7] (1983) to the more recent "Ontology-Informed Machine Learning" [3] (2023), the theme of how to embed and leverage knowledge for visual understanding has been consistently important, adapting with technological advancements.
*   **Machine Learning as a Core Methodology:** While only explicitly mentioned in a few papers ([3], [10]), machine learning, particularly deep learning, is the dominant underlying methodology driving many of the recent advancements and proposed directions in computer vision.
*   **Interdisciplinary Connections and Applications:** The papers frequently highlight the interdisciplinary nature of computer vision, with strong ties to robotics ([5], [9], [12]), computer graphics [4], information technology systems [6], and human-computer interaction [13], demonstrating its role as an enabling technology across various fields.
*   **Continuous Research and Development:** The recurring appearance of major conference proceedings (e.g., [1], [2], [8]) signifies the vibrant and continuously evolving nature of computer vision research, with new "developments" constantly emerging.

In conclusion, while many of the retrieved papers provide a historical perspective, the more recent publications indicate a shift towards integrating advanced AI techniques (like ontology-informed ML and visual reasoning) and focusing on dynamic, interactive perception (active vision) as key research directions and challenges.

## QUERY: What are the main research areas in computer science today, and how do emerging topics differ from more established ones?

The provided papers offer a diverse perspective on various aspects of computer science, though a significant portion focuses on education, history, and the broader impact of the field rather than directly enumerating current research areas or explicitly contrasting emerging with established topics.

Here's a summary of the key findings, contributions, and common themes:

**Key Findings and Contributions:**

1.  **Trends in Computer Science Research:**
    *   One recent paper [1] directly addresses "Trends in computer science research within European countries," indicating an ongoing effort to track the evolution of research domains. While the specific trends are not detailed in the title, its 2022 publication suggests an analysis of contemporary developments.
    *   Another paper [12] highlights "Research Software Science" and the expansion of "Research Software Engineering" as an area aiming to amplify the impact of research software, suggesting an emerging field focused on methodology and impact within scientific computing.

2.  **Measuring the Impact and Innovation of CS Research:**
    *   Papers explore how computer science research translates into real-world innovation and development [2]. This involves understanding the societal and economic contributions of the field.
    *   Relatedly, there's a focus on challenges to national competitiveness in the computer industry [8], hinting at the strategic importance of CS research outcomes for economic leadership.

3.  **Computer Science Education (CSEd) and Curriculum Development:**
    *   A prominent theme is the design and implementation of computer science curricula at various educational levels. This includes introducing concepts in undergraduate courses (CS2 projects) [3], developing curricula for secondary education [7], and integrating CS topics into first-year writing seminars [4].
    *   There's a significant interest in strategies for "Broadening Participation in Computer Science" [13] and understanding factors contributing to success in CS education policy [15], especially at a regional level (e.g., California).
    *   Conferences like ITiCSE-WGR [10] further demonstrate an active community dedicated to "Innovation and technology in computer science education."

4.  **Historical and Regional Development of Computer Science:**
    *   Several papers provide historical perspectives, such as "The First Decade of Computer Science in Argentina" [5] and "The history and contribution of theoretical computer science" [9]. These contribute to understanding the origins, growth, and foundational elements of the field and its sub-disciplines.
    *   Challenges in program development are also noted historically [14].

**Common Themes:**

*   **Education is a Central Concern:** The most dominant theme across these papers is Computer Science Education (CSEd). This includes curriculum design [3, 7], pedagogical approaches [3, 4], issues of access and participation [13], and policy considerations [15]. The frequent appearance of "ACM SIGCSE Bull." as a venue (e.g., [3], [7], [14]) underscores this focus.
*   **Impact and Relevance:** Several papers touch upon the impact of computer science, whether it's measuring the translation of research into innovation [2], its role in industry competitiveness [8], or the broader societal implications of research software [12].
*   **Evolution and History:** The historical development of CS as a discipline, both generally and in specific contexts (theoretical CS, regional growth), is a recurring theme [5, 9, 14].
*   **Identifying Trends (Limited):** While the query specifically asks about trends, only paper [1] directly addresses "Trends in computer science research." Paper [12] points to Research Software Science as an expanding area, hinting at an emerging focus.

**Methodologies (as inferred from titles):**

*   **Quantitative Analysis/Measurement:** Implied by papers like [1] (trends analysis) and [2] (measuring translation into innovation).
*   **Historical Analysis/Case Studies:** Evident in papers discussing the history of CS in specific regions or subfields [5, 9, 15].
*   **Curriculum Development and Pedagogical Design:** Explicit in papers proposing new curricula or teaching methods [3, 7].
*   **Panel Discussions and Working Group Reports:** Indicative of collaborative qualitative discussions and synthesis of ideas, particularly in education [4, 10].
*   **Policy Analysis:** Used to examine factors influencing success in CS education policy [15].

In conclusion, while the retrieved papers offer valuable insights into the **historical evolution, educational challenges, and societal impact** of computer science, they provide limited direct information on the comprehensive "main research areas in computer science today" or an explicit differentiation between "emerging topics" and "established ones," beyond specific mentions like "Research Software Science" [12] and general "trends" analysis [1]. The strong emphasis on education policy and curriculum suggests that nurturing the next generation of computer scientists and ensuring accessible, relevant education remains a significant focus within the broader computer science community.